# Gray-Scott with video and saving spatiotemporal output



## Model equations and setup

We simulate the 2D Gray-Scott reaction-diffusion system for two species $u(x,y,t)$ and $v(x,y,t)$:

$$
\frac{\partial u}{\partial t} = D_u \nabla^2 u - u v^2 + F(1-u),
$$
$$
\frac{\partial v}{\partial t} = D_v \nabla^2 v + u v^2 - (F+k)v.
$$

In this notebook, `GrayScott` uses:
- Feed and kill parameters from the selected `pattern` (`"spirals"`, `"gliders"`) unless explicitly overridden.
- Diffusion coefficients fixed by default to $D_u = 2\times 10^{-5}$ and $D_v = 1\times 10^{-5}$.

### Initial condition (this notebook)

The notebook sets `initial_condition="gaussians"`. In that mode:
- Two random smooth fields are generated from sums of Gaussian bumps (periodic image copies included).
- The fields are normalized to $[0,1]$.
- The simulator initializes
$$
u_0(x,y)=1-\text{base}_u(x,y), \quad v_0(x,y)=\text{base}_v(x,y).
$$

So $u$ starts near 1 with localized depletion, and $v$ starts near 0 with localized activation patches.

### Boundary conditions

Periodic boundary conditions are used in both spatial directions (spectral FFT solver on a periodic square domain):
$$
u(x+L,y,t)=u(x,y,t), \quad u(x,y+L,t)=u(x,y,t),
$$
$$
v(x+L,y,t)=v(x,y,t), \quad v(x,y+L,t)=v(x,y,t).
$$

In [ ]:
import torch

from autosim.utils import plot_spatiotemporal_video


In [ ]:
from autosim.simulations import GrayScott as Sim

In [ ]:
# Run simulations and save outputs in a dictionary
outputs = {}
for pattern in ["spirals", "gliders"]:
    sim = Sim(
        return_timeseries=True,
        n=64,
        L=1.0,
        # T=300.0,
        # dt=1.0,
        # snapshot_dt=1.0,
        T=1280.0,
        dt=1.0,
        snapshot_dt=4.0,
        initial_condition="gaussians",
        # initial_gaussian_spec={
        #   "count": (2, 10), "amplitude": (1.0, 3.0), "width": (150.0, 300.0)
        # },
        pattern=pattern,
    )

    train = sim.forward_samples_spatiotemporal(n=200, random_seed=None)
    valid = sim.forward_samples_spatiotemporal(n=20, random_seed=None)
    test = sim.forward_samples_spatiotemporal(n=20, random_seed=None)
    outputs[pattern] = {"train": train, "valid": valid, "test": test}

In [ ]:
# Example sample params
sim.sample_inputs(n_samples=5)

In [ ]:

import os

for pattern in ["spirals", "gliders"]:
    for split in ["train", "valid", "test"]:
        os.makedirs(f"../../autocast/datasets/gray_scott_range_{pattern}/{split}/", exist_ok=True)
        torch.save(outputs[pattern][split], f"../../autocast/datasets/gray_scott_range_{pattern}/{split}/data.pt")

In [ ]:
anim = plot_spatiotemporal_video(outputs["spirals"]["train"]["data"], batch_idx=0)

In [ ]:
from IPython.display import HTML

HTML(anim.to_jshtml())